In [ ]:
# Data manipulation and scientific computingimport pandas as pdimport numpy as npimport os# Audio processing librariesimport librosaimport soundfile as sfimport scipy.signal# Machine Learning and Deep Learningimport tensorflow as tffrom tensorflow import kerasfrom tensorflow.keras import layersfrom sklearn.model_selection import (    train_test_split,     cross_val_score,     GridSearchCV)from sklearn.preprocessing import (    StandardScaler,     LabelEncoder)from sklearn.pipeline import Pipeline# Machine Learning Modelsfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.svm import SVCfrom sklearn.naive_bayes import GaussianNBfrom sklearn.neural_network import MLPClassifier# Evaluation Metricsfrom sklearn.metrics import (    accuracy_score,    precision_score,    recall_score,    f1_score,    confusion_matrix,    classification_report,    roc_auc_score)# Visualizationimport matplotlib.pyplot as pltimport seaborn as sns# Additional utilitiesimport warningswarnings.filterwarnings('ignore')# Set random seeds for reproducibilitynp.random.seed(42)tf.random.set_seed(42)print("Environment setup complete.")

In [ ]:
# Define base paths for datasetBASE_PATH = r"G:\Msc\NCU\Doctoral Record\multimodal_medical_diagnosis\data\recordings"# Paths for different dataset componentsPATHS = {    'csv': os.path.join(BASE_PATH, 'overview-of-recordings.csv'),    'test_audio': os.path.join(BASE_PATH, 'test'),    'train_audio': os.path.join(BASE_PATH, 'train'),    'validate_audio': os.path.join(BASE_PATH, 'validate')}def load_medical_dataset(csv_path):    """    Load medical dataset metadata from CSV file        Args:        csv_path (str): Path to the CSV file        Returns:        pd.DataFrame: Loaded dataset metadata    """    try:        # Load dataset        df = pd.read_csv(csv_path)                # Basic initial preprocessing        df.dropna(subset=['Filename', 'Prompt'], inplace=True)                print("Dataset Metadata Loaded Successfully!")        print(f"Dataset Shape: {df.shape}")        print(f"Columns: {list(df.columns)}")        print(f"Unique Prompts: {df['Prompt'].nunique()}")                return df        except Exception as e:        print(f"Error loading dataset: {e}")        return None# Load dataset metadatadf = load_medical_dataset(PATHS['csv'])

In [ ]:
def load_audio_file(file_path, sample_rate=16000):    """    Load and preprocess audio file        Args:        file_path (str): Path to audio file        sample_rate (int): Target sample rate        Returns:        numpy.ndarray: Preprocessed audio time series    """    try:        # Load audio file        audio, sr = librosa.load(file_path, sr=sample_rate)                # Noise reduction using spectral gating        noise_reduced = librosa.effects.hpss(audio)[0]                # Normalize audio        normalized_audio = librosa.util.normalize(noise_reduced)                return normalized_audio    except Exception as e:        print(f"Error processing {file_path}: {e}")        return Nonedef extract_audio_features(audio_data, sample_rate=16000):    """    Extract comprehensive audio features        Args:        audio_data (numpy.ndarray): Audio time series        sample_rate (int): Sampling rate of audio        Returns:        dict: Extracted audio features    """    # Ensure audio data is not None    if audio_data is None:        return None        features = {}        # 1. Mel-Frequency Cepstral Coefficients (MFCCs)    mfccs = librosa.feature.mfcc(y=audio_data, sr=sample_rate, n_mfcc=13)    features['mfccs_mean'] = np.mean(mfccs.T, axis=0)    features['mfccs_std'] = np.std(mfccs.T, axis=0)        # 2. Spectral Features    spectral_centroids = librosa.feature.spectral_centroid(y=audio_data, sr=sample_rate)    features['spectral_centroid_mean'] = np.mean(spectral_centroids)        # 3. Chroma Features    chroma = librosa.feature.chroma_stft(y=audio_data, sr=sample_rate)    features['chroma_mean'] = np.mean(chroma.T, axis=0)        # 4. Mel Spectrogram    mel_spectrogram = librosa.feature.melspectrogram(y=audio_data, sr=sample_rate)    features['mel_spectrogram_mean'] = np.mean(mel_spectrogram.T, axis=0)        return featuresdef process_audio_dataset(df, audio_dir):    """    Process entire audio dataset and extract features        Args:        df (pd.DataFrame): Dataset metadata        audio_dir (str): Directory containing audio files        Returns:        tuple: Processed features and labels    """    features_list = []    labels = []        for _, row in df.iterrows():        # Construct full file path        file_path = os.path.join(audio_dir, row['Filename'])                # Load and process audio        audio_data = load_audio_file(file_path)        audio_features = extract_audio_features(audio_data)                if audio_features is not None:            # Flatten features            flat_features = np.concatenate([                audio_features['mfccs_mean'],                audio_features['mfccs_std'],                [audio_features['spectral_centroid_mean']],                audio_features['chroma_mean'],                audio_features['mel_spectrogram_mean']            ])                        features_list.append(flat_features)            labels.append(row['Prompt'])        return np.array(features_list), np.array(labels)# Process training audio datasetX_train, y_train = process_audio_dataset(    df[df['Dataset'] == 'train'],     PATHS['train_audio'])# Process validation audio datasetX_val, y_val = process_audio_dataset(    df[df['Dataset'] == 'validate'],     PATHS['validate_audio'])# Process test audio datasetX_test, y_test = process_audio_dataset(    df[df['Dataset'] == 'test'],     PATHS['test_audio'])

In [ ]:
def perform_audio_eda(X_train, y_train):    """    Perform comprehensive audio feature exploratory data analysis        Args:        X_train (numpy.ndarray): Training features        y_train (numpy.ndarray): Training labels    """    # Class distribution    plt.figure(figsize=(12, 5))    plt.subplot(121)    unique, counts = np.unique(y_train, return_counts=True)    plt.bar(unique, counts)    plt.title('Class Distribution')    plt.xlabel('Prompt/Class')    plt.ylabel('Frequency')    plt.xticks(rotation=45)        # Feature distribution    plt.subplot(122)    plt.boxplot(X_train)    plt.title('Feature Distribution')    plt.xlabel('Features')    plt.ylabel('Values')    plt.tight_layout()    plt.show()        # Print feature statistics    print("\nFeature Statistics:")    print(pd.DataFrame(X_train).describe())# Perform EDAperform_audio_eda(X_train, y_train)

In [ ]:
def preprocess_features(X_train, X_val, X_test):    """    Preprocess and normalize audio features        Args:        X_train (numpy.ndarray): Training features        X_val (numpy.ndarray): Validation features        X_test (numpy.ndarray): Test features        Returns:        tuple: Preprocessed train, validation, and test features    """    # Initialize standard scaler    scaler = StandardScaler()        # Fit and transform training data    X_train_scaled = scaler.fit_transform(X_train)        # Transform validation and test data    X_val_scaled = scaler.transform(X_val)    X_test_scaled = scaler.transform(X_test)        return X_train_scaled, X_val_scaled, X_test_scaled# Preprocess featuresX_train_scaled, X_val_scaled, X_test_scaled = preprocess_features(    X_train, X_val, X_test)# Encode labelslabel_encoder = LabelEncoder()y_train_encoded = label_encoder.fit_transform(y_train)y_val_encoded = label_encoder.transform(y_val)y_test_encoded = label_encoder.transform(y_test)

In [ ]:
def train_and_evaluate_models(X_train, y_train, X_test, y_test):    """    Train and evaluate multiple machine learning models        Args:        X_train (numpy.ndarray): Training features        y_train (numpy.ndarray): Training labels        X_test (numpy.ndarray): Test features        y_test (numpy.ndarray): Test labels        Returns:        dict: Model performance metrics    """    # Define models    models = {        'Random Forest': RandomForestClassifier(),        'Support Vector Machine': SVC(probability=True),        'Naive Bayes': GaussianNB(),        'Neural Network': MLPClassifier(max_iter=1000)    }        # Deep Learning Model    def create_deep_learning_model(input_shape, num_classes):        model = keras.Sequential([            layers.Input(shape=(input_shape,)),            layers.Dense(64, activation='relu'),            layers.Dropout(0.3),            layers.Dense(32, activation='relu'),            layers.Dropout(0.2),            layers.Dense(num_classes, activation='softmax')        ])        model.compile(            optimizer='adam',            loss='sparse_categorical_crossentropy',            metrics=['accuracy']        )        return model        # Results storage    results = {}        # Traditional Machine Learning Models    for name, model in models.items():        # Cross-validation        cv_scores = cross_val_score(model, X_train, y_train, cv=5)                # Fit model        model.fit(X_train, y_train)                # Predictions        y_pred = model.predict(X_test)                # Metrics        results[name] = {            'Accuracy': accuracy_score(y_test, y_pred),            'Precision': precision_score(y_test, y_pred, average='weighted'),            'Recall': recall_score(y_test, y_pred, average='weighted'),            'F1 Score': f1_score(y_test, y_pred, average='weighted'),            'Cross-Val Score': cv_scores.mean()        }        # Deep Learning Model    dl_model = create_deep_learning_model(        X_train.shape[1],         len(np.unique(y_train))    )        # Train deep learning model    dl_history = dl_model.fit(        X_train, y_train,         validation_data=(X_val, y_val),        epochs=50,         batch_size=32,        verbose=0    )        # Evaluate deep learning model    dl_pred = dl_model.predict(X_test)    dl_pred_classes = np.argmax(dl_pred, axis=1)        results['Deep Learning'] = {        'Accuracy': accuracy_score(y_test, dl_pred_classes),        'Precision': precision_score(y_test, dl_pred_classes, average='weighted'),        'Recall': recall_score(y_test, dl_pred_classes, average='weighted'),        'F1 Score': f1_score(y_test, dl_pred_classes, average='weighted')    }        return results# Run model evaluationmodel_results = train_and_evaluate_models(    X_train_scaled, y_train_encoded,     X_test_scaled, y_test_encoded)# Display resultsfor model, metrics in model_results.items():    print(f"\n{model} Performance:")    for metric, value in metrics.items():        print(f"{metric}: {value:.4f}")

In [ ]:
def visualize_model_performance(results):    """    Visualize model performance metrics        Args:        results (dict): Model performance metrics    """    # Prepare data for plotting    metrics_df = pd.DataFrame.from_dict(results, orient='index')        # Plot    plt.figure(figsize=(12, 6))    metrics_df[['Accuracy', 'Precision', 'Recall', 'F1 Score']].plot(kind='bar', rot=45)    plt.title('Audio Classification Model Performance')    plt.xlabel('Models')    plt.ylabel('Score')    plt.tight_layout()    plt.show()def validate_research_hypothesis(results):    """    Validate research hypotheses based on model performance        Args:        results (dict): Model performance metrics    """    print("\n--- Research Hypothesis Validation ---")        # Hypothesis threshold (suggest 0.75 as sufficient performance)    threshold = 0.75        # Check if audio classification meets decision support criteria    hypothesis_met = any(        metrics['Precision'] > threshold and         metrics['Recall'] > threshold        for metrics in results.values()    )        if hypothesis_met:        print("H2a Supported: Audio analysis provides sufficient precision and recall")        print("Recommendation: Suitable for provider decision support")    else:        print("H20 Supported: Audio analysis provides insufficient precision and recall")        print("Recommendation: Further model refinement needed")# Visualize resultsvisualize_model_performance(model_results)# Validate hypothesisvalidate_research_hypothesis(model_results)

In [ ]:
def main():    """    Main function to execute the entire audio classification workflow    """    # Load dataset metadata    df = load_medical_dataset(PATHS['csv'])        # Process audio datasets    X_train, y_train = process_audio_dataset(        df[df['Dataset'] == 'train'],         PATHS['train_audio']    )    X_val, y_val = process_audio_dataset(        df[df['Dataset'] == 'validate'],         PATHS['validate_audio']    )    X_test, y_test = process_audio_dataset(        df[df['Dataset'] == 'test'],         PATHS['test_audio']    )        # Preprocess features    X_train_scaled, X_val_scaled, X_test_scaled = preprocess_features(        X_train, X_val, X_test    )        # Encode labels    label_encoder = LabelEncoder()    y_train_encoded = label_encoder.fit_transform(y_train)    y_val_encoded = label_encoder.transform(y_val)    y_test_encoded = label_encoder.transform(y_test)        # Perform EDA    perform_audio_eda(X_train_scaled, y_train_encoded)        # Train and evaluate models    model_results = train_and_evaluate_models(        X_train_scaled, y_train_encoded,         X_test_scaled, y_test_encoded    )        # Visualize results    visualize_model_performance(model_results)        # Validate hypothesis    validate_research_hypothesis(model_results)# Execute main workflowif __name__ == '__main__':    main()

## Research Limitations and Future Work1. Limited to audio-based classification2. Potential bias in dataset3. Future work: Multimodal classification4. Explore advanced feature extraction techniques5. Collect more diverse medical symptom audio data## ConclusionComprehensive audio classification workflow for medical symptom analysis, addressing research questions and hypotheses through systematic approach.